# 🏢 Chapter 3 — Buildings on OSM: Categorical & Continuous Maps
## Python for Urban Analysis — Ghana Edition

---

> **Professor's note:**
> *"A city's built environment IS its buildings. The footprint layer alone tells you about density, land consumption, morphological texture, and — indirectly — about socio-economic structure."*

### Learning Objectives
1. Extract building footprints from OSM
2. Classify buildings by type (residential, commercial, etc.)
3. Map building area as a continuous variable (choropleth)
4. Explore height/floors data (where available)
5. Produce the **hero visualisation** — building footprints colour-coded by area on a dark basemap

### Ghana Context
Building height data in OSM is sparse for most African cities. We handle this gracefully: where height/floors are available we use them; otherwise we fall back to **footprint area**, which is a strong proxy for building importance and is well-mapped in Accra.

## 3.0 — Install & Configure

In [ ]:
!pip install -q osmnx geopandas contextily mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import contextily as cx
import mapclassify
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────
STUDY_AREA       = 'Accra, Ghana'
CRS_WEB_MERCATOR = 'EPSG:3857'
CRS_UTM_ACCRA    = 'EPSG:32630'
DARK_BG          = '#0e1117'
ACCENT_CMAP      = 'YlOrRd'
BUILDING_TAGS    = {'building': True}

DATA_DIR    = Path('data')
FIGURES_DIR = Path('outputs') / 'figures'
for d in [DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'📍 Study area: {STUDY_AREA}')

## 3.1 — Extract Building Footprints

The tag `building=True` matches **any** building on OSM — residential, commercial, industrial, or the generic `building=yes`.  This is the broadest possible query.

> **Tip:** OSMnx may return point nodes or linestrings alongside polygons. We filter to keep only `Polygon` and `MultiPolygon` geometries — these are the actual footprints.

In [ ]:
print('🏗  Pulling building footprints (this may take a minute) ...')
buildings_raw = ox.features_from_place(STUDY_AREA, tags=BUILDING_TAGS)
print(f'   Raw features returned: {len(buildings_raw)}')

# Keep only polygon/multipolygon geometries
buildings = buildings_raw[
    buildings_raw.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].copy()
print(f'   Polygon footprints  : {len(buildings)}')

## 3.2 — Compute Footprint Area

We project to **UTM Zone 30N** (EPSG:32630) — a metre-based CRS that covers Accra — before computing area. Never compute area in EPSG:4326 (lat/lon)!

In [ ]:
buildings_proj = buildings.to_crs(CRS_UTM_ACCRA)
buildings['area_m2'] = buildings_proj.geometry.area.round(1)

print('── Footprint area statistics ──')
print(buildings['area_m2'].describe().to_string())

## 3.3 — Classify Buildings by Type

The OSM `building` tag values reveal function:

| Tag Value | Meaning | Typical in Accra |
|-----------|---------|------------------|
| `yes` | Generic (unmapped type) | Most buildings |
| `residential` | Housing | Compound houses, apartments |
| `commercial` | Offices/business | CBD area |
| `industrial` | Factories/warehouses | Tema industrial zone |
| `retail` | Shops/malls | Accra Mall, West Hills |
| `school` | Educational | — |
| `church` | Religious | — |

In [ ]:
if 'building' in buildings.columns:
    buildings['bldg_type'] = buildings['building'].fillna('unknown')
    print('── Building type breakdown ──')
    type_counts = buildings['bldg_type'].value_counts().head(15)
    for btype, n in type_counts.items():
        print(f'   {btype:25s}  {n:>6}')

## 3.4 — Explore Height & Floors Data

> **Reality check for Africa:** Building height (`height`) and floor count (`building:levels`) tags are very sparse in most African cities. This is normal — not a limitation. Footprint area is the most universally available building metric.

In [ ]:
for col in ['height', 'building:levels']:
    if col in buildings.columns:
        available = buildings[col].notna().sum()
        pct = 100 * available / len(buildings)
        print(f'   {col}: {available} records ({pct:.1f}%)')
    else:
        print(f'   {col}: not present in data')

# Convert to numeric where possible
if 'building:levels' in buildings.columns:
    buildings['levels'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
else:
    buildings['levels'] = np.nan

if 'height' in buildings.columns:
    buildings['height_m'] = pd.to_numeric(buildings['height'], errors='coerce')
else:
    buildings['height_m'] = np.nan

## 3.5 — Save Processed Buildings

In [ ]:
save_cols = [c for c in ['geometry', 'area_m2', 'bldg_type', 'levels', 'height_m'] if c in buildings.columns]
buildings_save = buildings[save_cols].copy()
buildings_path = DATA_DIR / 'buildings.gpkg'
buildings_save.to_file(buildings_path, driver='GPKG')
print(f'💾 Buildings saved → {buildings_path}')

## 3.6 — 🌟 HERO VISUALISATION: Building Footprints by Area

This is the signature output — the map that looks like the example image:
- **Dark basemap** (CartoDB DarkMatter)
- **Building footprints** colour-coded by area using `YlOrRd` (Yellow → Orange → Red)
- **White boundary** outline
- **Quantile classification** (7 equal-count classes)

> **Professor's note:** *"This single map tells more about urban structure than pages of text. Dense, small footprints = informal settlements. Large, regular footprints = institutional or commercial areas. The colour gradient makes the pattern instantly visible."*

In [ ]:
print('🎨 Creating building footprint map (area choropleth) ...')

boundary = ox.geocode_to_gdf(STUDY_AREA)

# Filter out extremely large polygons (likely mapping artifacts)
q99 = buildings['area_m2'].quantile(0.99)
bldg_plot = buildings[buildings['area_m2'] <= q99].copy()

fig, ax = plt.subplots(1, 1, figsize=(16, 16))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

# Plot building footprints coloured by area
bldg_plot.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax,
    column='area_m2',
    cmap=ACCENT_CMAP,
    scheme='quantiles',
    k=7,
    linewidth=0.1,
    edgecolor='#333333',
    alpha=0.9,
    legend=False,
)

# Boundary outline
boundary.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10
)

# Dark basemap
cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=14)

ax.set_axis_off()
ax.set_title(
    f'Building Footprints by Area — {STUDY_AREA}',
    fontsize=18, fontweight='bold', color='white', pad=20,
)

# Custom colour bar
sm = plt.cm.ScalarMappable(
    cmap=ACCENT_CMAP,
    norm=mcolors.Normalize(vmin=bldg_plot['area_m2'].min(), vmax=bldg_plot['area_m2'].max()),
)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.6)
cbar.set_label('Footprint Area (m²)', color='white', fontsize=11)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch3_buildings_area.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 3.7 — Categorical Building Type Map

Now we colour buildings by their **function** rather than size. This reveals the spatial organisation of land use.

In [ ]:
type_palette = {
    'yes':           '#ffc107',
    'residential':   '#4caf50',
    'commercial':    '#2196f3',
    'industrial':    '#ff5722',
    'retail':        '#e91e63',
    'church':        '#9c27b0',
    'school':        '#00bcd4',
    'mosque':        '#8bc34a',
    'hospital':      '#f44336',
    'warehouse':     '#795548',
}

if 'bldg_type' in bldg_plot.columns:
    bldg_plot['color'] = bldg_plot['bldg_type'].map(type_palette).fillna('#757575')

    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    fig.patch.set_facecolor(DARK_BG)
    ax.set_facecolor(DARK_BG)

    bldg_plot.to_crs(CRS_WEB_MERCATOR).plot(
        ax=ax, color=bldg_plot['color'],
        linewidth=0.1, edgecolor='#222222', alpha=0.85,
    )

    boundary.to_crs(CRS_WEB_MERCATOR).plot(
        ax=ax, facecolor='none', edgecolor='white', linewidth=2.5, zorder=10
    )

    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=14)
    ax.set_axis_off()
    ax.set_title(
        f'Building Types — {STUDY_AREA}',
        fontsize=18, fontweight='bold', color='white', pad=20,
    )

    handles = [mpatches.Patch(color=c, label=t) for t, c in type_palette.items()]
    ax.legend(
        handles=handles, loc='lower left', fontsize=8,
        facecolor='#1a1a2e', edgecolor='white',
        labelcolor='white', framealpha=0.9, ncol=2
    )

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'ch3_buildings_type.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()

## ✅ Chapter 3 Complete

### What you produced:
- **Building area choropleth** — the hero image (YlOrRd on dark basemap)
- **Categorical type map** — buildings by function
- **Statistics** on footprint area, height/floors availability

### Key takeaway:
> Building footprint area is the most universally available and revealing metric for urban morphology. Even without height data, the footprint layer tells you about density, scale, and function.

### Next → Chapter 4: Road Networks